In [14]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
@tool
def multiply(a:int, b:int) -> int:
    '''given 2 numbers a and b this tool returns their product'''
    return a * b

In [3]:
print(multiply.invoke({"a" : 4, "b" : 5}))

20


In [4]:
multiply.name

'multiply'

In [5]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [6]:
multiply.description

'given 2 numbers a and b this tool returns their product'

In [15]:
model = ChatOpenAI(model = 'gpt-5.5', temperature = 0)

In [16]:
llm_with_tools = model.bind_tools([multiply])

In [18]:
llm_with_tools.invoke("Hi").content

'Hi! How can I help you today?'

In [19]:
query = HumanMessage("can you multiply 3 with 1000")

In [20]:
messages = [query]
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={})]

In [21]:
result = llm_with_tools.invoke(messages)
messages.append(result)
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 143, 'total_tokens': 164, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.5-2026-04-23', 'system_fingerprint': None, 'id': 'chatcmpl-EADzQboRRfICxmIAQJkm4nYR9Endl', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fdc2f-ded8-7940-a9a4-6f409c23d4b3-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'call_8ndRJTMveePtkhHijnXh8Jf4', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 143, 'output_tokens': 21, 'total_tokens': 164, 'input_token_details': {'audio': 0, 

In [23]:
tool_result = multiply.invoke(result.tool_calls[0])

In [24]:
messages.append(tool_result)

In [25]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 143, 'total_tokens': 164, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.5-2026-04-23', 'system_fingerprint': None, 'id': 'chatcmpl-EADzQboRRfICxmIAQJkm4nYR9Endl', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fdc2f-ded8-7940-a9a4-6f409c23d4b3-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'call_8ndRJTMveePtkhHijnXh8Jf4', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 143, 'output_tokens': 21, 'total_tokens': 164, 'input_token_details': {'audio': 0, 

In [26]:
llm_with_tools.invoke(messages).content

'3000'

In [28]:
# tool create 
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency : str, target_currency : str) -> float:
    '''This function fetches the currency conversion factor between a given base currency and target currency'''
    url = f"https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}"
    response = requests.get(url)
    return response.json()


@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    '''given a currency conversion rate this function calculates the target currency value from a given base currency value'''
    return base_currency_value * conversion_rate

In [29]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [30]:
get_conversion_factor.invoke({"base_currency": "USD", "target_currency" : "INR"})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1786060801,
 'time_last_update_utc': 'Fri, 07 Aug 2026 00:00:01 +0000',
 'time_next_update_unix': 1786147201,
 'time_next_update_utc': 'Sat, 08 Aug 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.265}

In [31]:
convert.invoke({"base_currency_value" : 10, "conversion_rate" : 95.265})

952.65

In [32]:
llm_with_tools = model.bind_tools([get_conversion_factor, convert])

In [33]:
messages = [HumanMessage("What is the conversion factor between INR and USD , and based on that can you convert 10 inr to usd")]

In [34]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD , and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={})]

In [35]:
ai_message = llm_with_tools.invoke(messages)

In [36]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': 'call_GSMBeLol8EsNDUSPNJJvCVmV',
  'type': 'tool_call'}]

In [38]:
import json

for tool_call in ai_message.tool_calls:
    # execute the 1st tool and get the value of conversion rate
    if tool_call['name'] == 'get_conversion_factor':
        tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    messages.append(tool_message1)

    # execute the 2nd tool using the conversion rate from tool 1
    if tool_call['name'] == 'convert':
        # fetch the current arg
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        messages.append(tool_message2)

In [39]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD , and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 ToolMessage(content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1786060801, "time_last_update_utc": "Fri, 07 Aug 2026 00:00:01 +0000", "time_next_update_unix": 1786147201, "time_next_update_utc": "Sat, 08 Aug 2026 00:00:01 +0000", "base_code": "INR", "target_code": "USD", "conversion_rate": 0.0105}', name='get_conversion_factor', tool_call_id='call_GSMBeLol8EsNDUSPNJJvCVmV')]

In [40]:
llm_with_tools.invoke(messages).content

BadRequestError: Error code: 400 - {'error': {'message': "Invalid parameter: messages with role 'tool' must be a response to a preceeding message with 'tool_calls'.", 'type': 'invalid_request_error', 'param': 'messages.[1].role', 'code': None}}

In [1]:
print("hello")

hello


In [8]:
import requests
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

load_dotenv()

# 1. Define Tools
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """Fetches the conversion rate between a base currency and a target currency."""
    url = f"https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}"
    response = requests.get(url)
    data = response.json()
    return data.get("conversion_rate", 0.0)

@tool
def convert(base_currency_value: float, conversion_rate: float) -> float:
    """Calculates the target currency value given a base value and conversion rate."""
    return base_currency_value * conversion_rate

# 2. Setup Model & Tools
model = ChatOpenAI(model="gpt-4o", temperature=0)
tools = [get_conversion_factor, convert]

# 3. Create Agent
agent_executor = create_react_agent(model, tools)

# 4. Invoke Agent
query = "What is the conversion factor between INR and USD, and based on that convert 10 USD to INR?"
response = agent_executor.invoke({"messages": [("user", query)]})

# Print Final Answer
print(response["messages"][-1].content)

C:\Users\Admin\AppData\Local\Temp\ipykernel_3676\1336049514.py:28: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(model, tools)


The conversion factor between USD and INR is 95.2421. Based on this rate, 10 USD is equivalent to approximately 952.42 INR.
